Cell 1: Mount Drive & Set Working Directory

In [ ]:
# Mount Google Drive to access the dataset stored in the cloud
from google.colab import drive
drive.mount('/content/drive')

# Switch to the project directory containing the datasets and the notebook
WORKING_DIR = '/content/drive/MyDrive/FTEC5560'
import os
os.chdir(WORKING_DIR)
print(f"Changed working directory to: {os.getcwd()}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Changed working directory to: /content/drive/MyDrive/FTEC5560


Cell 2: Install Libraries

In [ ]:
!pip install openai numpy pandas scipy matplotlib seaborn

Cell 3: Import Libraries & Define Configuration

In [ ]:
import random
import numpy as np
import pandas as pd
from openai import OpenAI
from scipy.stats import norm
import os
import glob
from google.colab import userdata

API_KEY_ENV = userdata.get('siliconflow_api_key')
BASE_URL = "https://api.siliconflow.cn/v1"

# Define the models, N-back difficulties, and temperature parameters to test
# MODEL_NAMES_TO_TEST = ["Qwen/Qwen3-32B", "deepseek-ai/DeepSeek-V3"]
MODEL_NAMES_TO_TEST = ["Qwen/Qwen3-32B"]
N_BACK_VALUES = [1, 2]
TEMPERATURES_TO_TEST = [0.7]
# NUM_BLOCKS_PER_N = 10 # Use data from 10 blocks
NUM_BLOCKS_PER_N = 1
TRIALS_PER_BLOCK = 30
# Specify the paths to the dataset folders
FIXED_DATASET_PATH_1BACK = "1back/"
FIXED_DATASET_PATH_2BACK = "2back/"

# Initialize the OpenAI client
client = OpenAI(api_key=API_KEY_ENV, base_url=BASE_URL)

Cell 4: Helper Functions

In [ ]:
def load_sequence_and_targets_from_file(file_path):
    """Load the letter sequence and target labels from a specified file."""
    with open(file_path, 'r') as f:
        content = f.read().strip()
    sequence_part = content[:TRIALS_PER_BLOCK]
    targets_part = content[TRIALS_PER_BLOCK:]
    sequence = list(sequence_part.upper())
    targets = list(targets_part.lower())
    return sequence, targets

def get_model_response(client, prompt, model_name, temperature=0.7):
    """Call the model API to get a response. This version does NOT handle timeouts."""
    # 直接调用 API，不使用 try-except 和 timeout
    completion = client.chat.completions.create(
        model=model_name,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=10,
        temperature=temperature,
        # timeout=timeout 参数被移除
    )
    # Get the raw text response from the model and convert to lowercase
    response_text = completion.choices[0].message.content.strip().lower()
    print(f"      Raw API Response: '{response_text}'") # Print the raw response

    # Improved parsing logic using regex to find standalone 'm' or '-'
    import re
    match_m = re.search(r'(?<!\w)m(?!\w)', response_text)
    match_dash = re.search(r'(?<!\w)-(?!\w)', response_text)

    if match_m:
        return 'm'
    elif match_dash:
        return '-'
    # Fallback: if no clear match, use the old method of finding first occurrence
    else:
        for char in response_text:
            if char == 'm':
                return 'm'
            elif char in ['-', 'n']:
                return '-'
        # If neither 'm' nor '-' is found, default to '-'
        return '-'

def calculate_metrics_per_block(targets, responses):
    """Calculate psychophysical metrics for a single block based on targets and responses."""
    hits = sum(1 for t, r in zip(targets, responses) if t == 'm' and r == 'm')
    misses = sum(1 for t, r in zip(targets, responses) if t == 'm' and r == '-')
    fas = sum(1 for t, r in zip(targets, responses) if t == '-' and r == 'm')
    crs = sum(1 for t, r in zip(targets, responses) if t == '-' and r == '-')

    total_signal = hits + misses
    total_noise = fas + crs

    hit_rate = hits / total_signal if total_signal > 0 else 0.0
    fa_rate = fas / total_noise if total_noise > 0 else 0.0

    accuracy = (hits + crs) / len(targets) if targets else 0.0

    epsilon = 1e-9
    corrected_hr = np.clip(hit_rate, epsilon, 1 - epsilon)
    corrected_far = np.clip(fa_rate, epsilon, 1 - epsilon)
    d_prime = norm.ppf(corrected_hr) - norm.ppf(corrected_far)

    return {
        'hit_rate': hit_rate, 'fa_rate': fa_rate,
        'accuracy': accuracy, 'd_prime': d_prime,
        'hits': hits, 'misses': misses, 'fas': fas, 'crs': crs
    }

Cell 5: Define Main Experiment Runner

In [ ]:
def run_single_condition(n_back, num_blocks, client, model_name, temp_value, condition_label=""):
    """Run a single experimental condition (e.g., Qwen 1-back) and collect results from all blocks."""
    print(f"Starting {condition_label}...")
    path = FIXED_DATASET_PATH_1BACK if n_back == 1 else FIXED_DATASET_PATH_2BACK
    # Get all .txt files, sort them, and take the first num_blocks
    all_files = sorted(glob.glob(os.path.join(path, "*.txt")))
    file_paths = all_files[:num_blocks]

    all_block_results = []
    for block_num, file_path in enumerate(file_paths):
        print(f"  Processing Block {block_num+1}/{len(file_paths)} from {file_path}...")
        sequence, targets = load_sequence_and_targets_from_file(file_path)

        responses = []
        for i, letter in enumerate(sequence):
            # Build the prompt containing the current letter sequence
            context_str = ", ".join(sequence[:i+1])
            prompt = f"N-back task, N is {n_back}. Respond 'm' if current letter matches the one {n_back} steps back, otherwise '-'. Sequence so far: {context_str}. Your response?"
            resp = get_model_response(client, prompt, model_name, temperature=temp_value)
            responses.append(resp)

        # Calculate metrics for the current block
        metrics = calculate_metrics_per_block(targets, responses)
        metrics['block_id'] = int(os.path.basename(file_path).split('.')[0]) # Use filename as block_id
        metrics['model'] = model_name
        metrics['n_back'] = n_back
        metrics['temperature'] = temp_value
        all_block_results.append(metrics)

    df = pd.DataFrame(all_block_results)
    print(f"Completed {condition_label}. Processed {len(df)} blocks.")
    return df

Cell 6: Execute Full Experiment & Save Results

In [ ]:
# Run all defined combinations of experimental conditions
all_results_dfs = []
for model_name in MODEL_NAMES_TO_TEST:
    for n_val in N_BACK_VALUES:
        for temp_val in TEMPERATURES_TO_TEST:
            condition_label = f"{model_name}_Fixed_T{temp_val}_{n_val}back"
            df = run_single_condition(
                n_back=n_val,
                num_blocks=NUM_BLOCKS_PER_N,
                client=client,
                model_name=model_name,
                temp_value=temp_val,
                condition_label=condition_label
            )
            all_results_dfs.append(df)

# Concatenate all results and save them to a CSV file
final_df = pd.concat(all_results_dfs, ignore_index=True)
print("\nFinal DataFrame shape:", final_df.shape)
print(final_df.head(10)) # Display the first 10 rows of data

# Save results to the current working directory
output_filename = "llm_nback_results.csv"
final_df.to_csv(output_filename, index=False)
print(f"\nResults saved to {output_filename}")

# Print a simple summary statistic
summary_stats = final_df.groupby(['model', 'n_back']).agg({
    'accuracy': ['mean', 'std'],
    'd_prime': ['mean', 'std']
}).round(4)

print("\nSummary Statistics (Mean and Std for Accuracy and d'):")
print(summary_stats)

Starting Qwen/Qwen3-32B_Fixed_T0.7_1back...
  Processing Block 1/1 from 1back/0.txt...
      Raw API Response: '-'
      Raw API Response: '-'
      Raw API Response: '-'
      Raw API Response: 'the current letter is the fourth in the sequence'
      Raw API Response: 'm'
      Raw API Response: 'the current letter (the sixth in the sequence'
      Raw API Response: 'the current letter is f, and the one'
      Raw API Response: '-'
      Raw API Response: 'm'
      Raw API Response: 'm'
      Raw API Response: 'the current letter is q, and the one step'
      Raw API Response: 'the current letter is r. the letter one step'
      Raw API Response: 'm'
      Raw API Response: '-'
      Raw API Response: 'the current letter is b, and the previous one'
      Raw API Response: '-'
      Raw API Response: 'm'
      Raw API Response: 'm'
      Raw API Response: 'the current letter is n. the letter one step'
      Raw API Response: '-'
      Raw API Response: '-'
      Raw API Response: 'the 